In [3]:
!pip install tensorboardX rdkit
!pip install rdkit-pypi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 56.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 56.9 MB/s eta 0:00:00:00:0100:01


In [12]:
import os
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as Data
torch.manual_seed(8) # for reproduce


from sklearn.model_selection import LeaveOneOut
import time
from tqdm import tqdm
import numpy as np
import gc
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Posdoc/Native_AFP/code/')
sys.setrecursionlimit(50000)
import pickle
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
# from tensorboardX import SummaryWriter
torch.nn.Module.dump_patches = True
import copy
import pandas as pd
#then import my own modules
from GCN import RelativeGCNModel,save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight
from rdkit import Chem
# from rdkit.Chem import AllChem
from rdkit.Chem import QED
from rdkit.Chem import rdMolDescriptors, MolSurf
from rdkit.Chem.Draw import SimilarityMaps
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
%matplotlib inline
from numpy.polynomial.polynomial import polyfit
import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib.cm as cm
import matplotlib
import seaborn as sns; sns.set_style("darkgrid")
from IPython.display import SVG, display
from torch.utils.data import Dataset, TensorDataset, DataLoader
import uuid

import itertools
from sklearn.metrics import r2_score
import scipy
from itertools import combinations
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import pickle
import os
import time
from rdkit import Chem
from sklearn.model_selection import LeaveOneOut
from collections import defaultdict


device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RelativeGCNModel(nn.Module):
    def __init__(self, radius, T, input_feature_dim, input_bond_dim,
                 fingerprint_dim, output_units_num, p_dropout):
        super(RelativeGCNModel, self).__init__()
        
        self.radius = radius
        self.T = T
        
        # Increase the dimensionality of hidden layers
        hidden_dim = fingerprint_dim * 4  # Significantly larger hidden dimension
        
        self.atom_fc = nn.Sequential(
            nn.Linear(input_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, fingerprint_dim)
        )
        
        self.neighbor_fc = nn.Sequential(
            nn.Linear(input_feature_dim + input_bond_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, fingerprint_dim)
        )
        
        # Increase the number and size of GCN layers
        self.gcn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fingerprint_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, fingerprint_dim)
            ) for _ in range(radius * 2)  # Double the number of GCN layers
        ])
        
        self.dropout = nn.Dropout(p=p_dropout)
        
        # More complex output layer for pairwise prediction
        self.output = nn.Sequential(
            nn.Linear(fingerprint_dim * 2, hidden_dim),  # Input is concatenated features of two molecules
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_units_num)
        )

    def forward(self, atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1,
                atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2):
        
        batch_size, mol_length, num_atom_feat = atom_list1.size()

        # Process first molecule
        atom_feature1, mol_feature1 = self.process_molecule(atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1)
        
        # Process second molecule
        atom_feature2, mol_feature2 = self.process_molecule(atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2)
        
        # Concatenate features of both molecules and predict pairwise difference
        pairwise_feature = torch.cat([mol_feature1, mol_feature2], dim=1)
        pairwise_prediction = self.output(self.dropout(pairwise_feature))
        
        return atom_feature1, atom_feature2, pairwise_prediction

    def process_molecule(self, atom_list, bond_list, atom_degree_list, bond_degree_list, atom_mask):
        batch_size, mol_length, num_atom_feat = atom_list.size()
        atom_mask = atom_mask.unsqueeze(2)
        
        #print("1. atom_list shape:", atom_list.shape)
        atom_feature = self.atom_fc(atom_list)
        #print("2. atom_feature shape after atom_fc:", atom_feature.shape)
        
        neighbor_feature = self.get_neighbor_feature(atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size, mol_length)
        #print("3. neighbor_feature shape:", neighbor_feature.shape)
        
        for i, layer in enumerate(self.gcn_layers):
            neighbor_feature = layer(neighbor_feature)
           # print(f"4. neighbor_feature shape after gcn layer {i}:", neighbor_feature.shape)
            atom_feature = atom_feature + neighbor_feature
            atom_feature = self.dropout(atom_feature)
        #    print(f"5. atom_feature shape after layer {i}:", atom_feature.shape)
        
        mol_feature = torch.sum(F.relu(atom_feature) * atom_mask, dim=1)
       # print("6. mol_feature shape:", mol_feature.shape)
        return atom_feature, mol_feature

    

    def get_neighbor_feature(self, atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size, mol_length):
        neighbor_feature = []
        for i in range(batch_size):
            atom_degrees = atom_degree_list[i]
            bond_degrees = bond_degree_list[i]
        
            # Ensure indices are within bounds
            atom_degrees = torch.clamp(atom_degrees, 0, mol_length - 1)
            bond_degrees = torch.clamp(bond_degrees, 0, bond_list.size(1) - 1)
        
        # No need to pad, use the original tensors
            atom_features = atom_list[i]
            bond_features = bond_list[i]
        
        # Gather neighbor features
            neighbor_atoms = atom_features[atom_degrees]
            neighbor_bonds = bond_features[bond_degrees]
            
        # Combine atom and bond features
            mol_neighbor_feature = torch.cat([neighbor_atoms, neighbor_bonds], dim=-1)
        
        # Apply neighbor_fc to each atom's neighborhood and sum
            mol_neighbor_feature = self.neighbor_fc(mol_neighbor_feature)
            mol_neighbor_feature = mol_neighbor_feature.sum(dim=1)
        
            neighbor_feature.append(mol_neighbor_feature)
    
        neighbor_feature = torch.stack(neighbor_feature, dim=0)
        return F.relu(neighbor_feature)
    


In [6]:
#?) pairs are directed or undirected?
#?) tergets correspond correctly? 
#?) 

def custom_cv_split(df):
    all_smiles = set()
    for pair in df['SMILES_pair']:
        smiles1, smiles2 = pair.split(',')
        all_smiles.add(smiles1)
        all_smiles.add(smiles2)
    
    n_splits = len(all_smiles)
    
    for test_smile in all_smiles:
        test_indices = []
        train_indices = []
        for idx, pair in enumerate(df['SMILES_pair']):
            smiles1, smiles2 = pair.split(',')
            if test_smile in (smiles1, smiles2):
                test_indices.append(idx)
            else:
                train_indices.append(idx)
        yield train_indices, test_indices, test_smile

#_____________________________________________________________________________________________________________________________________________ 
targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
def validate(model, val_df, feature_dicts, loss_function, device, batch_size, return_predictions=False, test_smile=None):
    model.eval()
    total_loss = 0
    all_predictions = []
    all_true_values = []
    result_dict = {}

    with torch.no_grad():
        for i in range(0, len(val_df), batch_size):
            batch_df = val_df.iloc[i:i+batch_size]
            
            smile_pairs = batch_df.SMILES_pair.values
            y_val = batch_df[targets].values
            
            smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
            
            x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
            x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
            
            x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
            x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
            x_mask1 = torch.Tensor(x_mask1).to(device)
            
            x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
            x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
            x_mask2 = torch.Tensor(x_mask2).to(device)
            
            output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                                 x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
            
            predictions = output_tuple[2]
            
            loss = loss_function(predictions, torch.Tensor(y_val).to(device))
            total_loss += loss.item()
            
            all_predictions.extend(predictions.cpu().numpy())
            all_true_values.extend(y_val)

            # Populate result_dict
            for j, smile_pair in enumerate(smile_pairs):
                smile1, smile2 = smile_pair.split(',')
                if smile1 == test_smile or smile2 == test_smile:
                    if test_smile not in result_dict:
                        result_dict[test_smile] = {}
                    result_dict[test_smile][smile_pair] = {}
                    for k, target in enumerate(targets):
                        result_dict[test_smile][smile_pair][target] = {
                            "actual": y_val[j][k],
                            "predicted": predictions[j][k].item()
                        }
    
    avg_loss = total_loss / (len(val_df) // batch_size + 1)
    r2 = r2_score(all_true_values, all_predictions)
    
    if return_predictions:
        if test_smile:
            return avg_loss, r2, all_true_values, all_predictions, result_dict
        else:
            print('NO TEST SMILE!!!!!')
            return avg_loss, r2, all_true_values, all_predictions, result_dict
    else:
        return avg_loss, r2



#_____________________________________________________________________________________________________________________________________________        

targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
def prepare_model_and_data_for_relative_learning(raw_filename, target_name='Calx', targets=None, random_seed=888, batch_size=50, epochs=800, p_dropout=0.1, fingerprint_dim=210, weight_decay=5, learning_rate=3, radius=4, T=2, per_target_output_units_num=1):
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    feature_filename = raw_filename.replace('.csv', '.pickle')
    smiles_targets_df = pd.read_csv(raw_filename)

    smilesList = smiles_targets_df.SMILES.values
    #print("number of all smiles: ", len(smilesList))

    atom_num_dist = []
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            atom_num_dist.append(len(mol.GetAtoms()))
            remained_smiles.append(smiles)
            canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
        except:
            print(smiles)
            pass
   # print("number of successfully processed smiles: ", len(remained_smiles))

    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = save_smiles_dicts(smilesList, filename)
    
    feature_dicts = get_smiles_dicts(smilesList)
    #print (type(feature_dicts))
    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    # Create pairs of SMILES and calculate relative targets
    smile_pairs = []
    relative_targets = []
    
    for i in range(len(remained_df)):
        for j in range(len(remained_df)):
            if i != j:  # Exclude self-pairs
                smile1 = remained_df.iloc[i]['cano_smiles']
                smile2 = remained_df.iloc[j]['cano_smiles']
                target1 = remained_df.iloc[i][targets].values
                target2 = remained_df.iloc[j][targets].values
                
                # Calculate relative target (difference)
                rel_target = target1 - target2
                
                smile_pairs.append(f"{smile1},{smile2}")
                relative_targets.append(rel_target)

    # Create DataFrame
    df = pd.DataFrame(relative_targets, columns=targets)
    df['SMILES_pair'] = smile_pairs
    df = df[['SMILES_pair'] + targets]  # Reorder columns

    # Prepare input features for the model
    feature_data = []
    
    for smile_pair in smile_pairs:
        smile1, smile2 = smile_pair.split(',')
        x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array([smile1], feature_dicts)
        x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array([smile2], feature_dicts)
        
        feature_data.append({
            'x_atom1': x_atom1,
            'x_bonds1': x_bonds1,
            'x_atom_index1': x_atom_index1,
            'x_bond_index1': x_bond_index1,
            'x_mask1': x_mask1,
            'x_atom2': x_atom2,
            'x_bonds2': x_bonds2,
            'x_atom_index2': x_atom_index2,
            'x_bond_index2': x_bond_index2,
            'x_mask2': x_mask2
        })

    num_atom_features = feature_data[0]['x_atom1'].shape[-1]
    num_bond_features = feature_data[0]['x_bonds1'].shape[-1]

    loss_function = nn.MSELoss()
    model = RelativeGCNModel(radius, T, num_atom_features, num_bond_features, fingerprint_dim, len(targets), p_dropout)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)

    return model, optimizer, loss_function, df, feature_data, feature_filename,remained_df

# Example usage:
model, optimizer, loss_function, df, feature_data, feature_filename ,remained_df= prepare_model_and_data_for_relative_learning('/notebooks/data/cal_abs.csv')


In [7]:
if os.path.isfile(feature_filename):
    feature_dicts = pickle.load(open(feature_filename, "rb" ))
else:
    feature_dicts = save_smiles_dicts(smilesList,filename)
    
# feature_dicts = get_smiles_dicts(smilesList)

print(type(feature_dicts))
for index, row in df.iterrows():
    # Split the SMILES pair
    smile1, smile2 = row['SMILES_pair'].split(',')
    
    # Get features for smile1
    x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array([smile1], feature_dicts)

    # Get features for smile2
    x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array([smile2], feature_dicts)
print(x_atom2.shape)

<class 'dict'>
(1, 73, 39)


In [17]:



def train_and_evaluate(model, df, feature_dicts, optimizer, loss_function, num_epochs=100, batch_size=32, patience=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    fold_results = []
    all_true_values = []
    all_predictions = []
    final_dic= {}
    test_mses = [] 
    initial_state = {k: v.clone() for k, v in model.state_dict().items()}
    
    n_splits = len(set(smiles for pair in df['SMILES_pair'] for smiles in pair.split(',')))
    
    for fold, (train_val_index, test_index, test_smile) in enumerate(custom_cv_split(df), 1):
        #print(f"\nFold {fold}/{n_splits}")
        #print(f"Testing SMILES: {test_smile}")
        
        # Reset the model for each fold
        model.load_state_dict(initial_state)
        
        # Reset optimizer state
        optimizer.state = defaultdict(dict)
        
        # Split the data into train+val and test
        train_val_df = df.iloc[train_val_index]
        test_df = df.iloc[test_index]
        
        # Check for test SMILES in train_val set
        train_val_smiles = set()
        for pair in train_val_df['SMILES_pair']:
            smiles1, smiles2 = pair.split(',')
            train_val_smiles.add(smiles1)
            train_val_smiles.add(smiles2)
        
        if test_smile in train_val_smiles:
            print(f"WARNING: Test SMILES {test_smile} found in train_val set!")
        #else:
            #print(f"Verification: Test SMILES {test_smile} not found in train_val set.")
        
        # Further split train_val into train and validation
        train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)
        
        # Additional check for test SMILES in train and val sets
        train_smiles = set()
        for pair in train_df['SMILES_pair']:
            smiles1, smiles2 = pair.split(',')
            train_smiles.add(smiles1)
            train_smiles.add(smiles2)
        
        val_smiles = set()
        for pair in val_df['SMILES_pair']:
            smiles1, smiles2 = pair.split(',')
            val_smiles.add(smiles1)
            val_smiles.add(smiles2)
        
        if test_smile in train_smiles:
            print(f"WARNING: Test SMILES {test_smile} found in training set!")
        #else:
            #print(f"Verification: Test SMILES {test_smile} not found in training set.")
        
        if test_smile in val_smiles:
            print(f"WARNING: Test SMILES {test_smile} found in validation set!")
        #else:
            #print(f"Verification: Test SMILES {test_smile} not found in validation set.")
        
        best_val_loss = float('inf')
        epochs_no_improve = 0
        
        for epoch in range(num_epochs):
            # Training code (unchanged)
        

            model.train()
            total_loss = 0
            
            train_df = train_df.sample(frac=1).reset_index(drop=True)
            
            for i in range(0, len(train_df), batch_size):
                batch_df = train_df.iloc[i:i+batch_size]
                
                smile_pairs = batch_df.SMILES_pair.values
                y_val = batch_df[targets].values
                
                smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
                
                x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
                x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
                
                x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
                x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
                x_mask1 = torch.Tensor(x_mask1).to(device)
                
                x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
                x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
                x_mask2 = torch.Tensor(x_mask2).to(device)
                
                output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                                     x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
                
                predictions = output_tuple[2]
                
                optimizer.zero_grad()
                loss = loss_function(predictions, torch.Tensor(y_val).to(device))
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            avg_loss = total_loss / (len(train_df) // batch_size + 1)
            print(f'Epoch {epoch+1}/{num_epochs} done. Average Loss: {avg_loss}')
            
            # Validation step
            val_loss, val_r2 = validate(model, val_df, feature_dicts, loss_function, device, batch_size)
            print(f'Validation Loss: {val_loss}, Validation R2: {val_r2}')
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                #print(f'New best model found for fold {fold} with validation loss: {best_val_loss}')
                # Save the best model params as a pth file
                unique_id = str(uuid.uuid4())[:8]
                torch.save(model.state_dict(), f'/notebooks/GNN/saved_models/best_model_fold_{fold}_{unique_id}.pth')
            else:
                epochs_no_improve += 1
                
            if epochs_no_improve == patience:
                #print(f'Early stopping triggered after {epoch+1} epochs')
                break
        

        # Evaluate on the test set for this fold
        model.load_state_dict(torch.load(f'/notebooks/GNN/saved_models/best_model_fold_{fold}_{unique_id}.pth'))
        
        test_loss, test_r2, true_values, predictions, result_dict = validate(model, test_df, feature_dicts, loss_function, device, batch_size, return_predictions=True, test_smile=test_smile)
        
        # Calculate MSE for this fold
        fold_mse = mean_squared_error(true_values, predictions)
        test_mses.append(fold_mse)

        final_dic.update(result_dict)
        
        all_true_values.extend(true_values)
        all_predictions.extend(predictions)
        
        fold_results.append((best_val_loss, test_loss, test_r2, fold_mse))  
     #   print(f'Fold {fold} completed. Best validation loss: {best_val_loss}, Test loss: {test_loss}, Test R2: {test_r2}')
    
    #print('\nCross-validation completed.')
    overall_r2 = r2_score(all_true_values, all_predictions)
    overall_mse = mean_squared_error(all_true_values, all_predictions)

    return fold_results, overall_r2, final_dic, test_r2, test_mses, overall_mse

# Usage
#fold_results, overall_r2, final_dic, test_r2s, test_mses, overall_mse = train_and_evaluate(model, df, feature_dicts, optimizer, loss_function, num_epochs=800, batch_size=32, patience=30)



In [18]:
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_squared_error

def random_search(param_distributions, n_iter):
    param_list = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=42))
    return param_list

def run_random_search(raw_filename, n_iter=10):
    # Define the hyperparameter space
    param_distributions = {
        'radius': [2, 3, 4, 5, 6],
        'fingerprint_dim': [30, 70, 150, 210, 300],
        'p_dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
        'weight_decay': [2, 3, 4, 5, 6],
        'learning_rate': [2, 3, 4, 5]
    }

    # Fixed parameters
    fixed_params = {
        'T': 2,
        'batch_size': 32,
        'epochs': 2,
        'patience': 30
    }

    # Generate random hyperparameter combinations
    param_list = random_search(param_distributions, n_iter)

    best_mse = float('inf')
    best_params = None

    for params in param_list:
        print(f"Testing parameters: {params}")

        # Combine fixed and variable parameters
        current_params = {**fixed_params, **params}

        # Separate parameters for data preparation and training
        data_prep_params = {k: v for k, v in current_params.items() if k not in ['patience', 'epochs', 'batch_size']}
        
        # Prepare model and data with current hyperparameters
        model, optimizer, loss_function, df, feature_data, feature_filename, remained_df = prepare_model_and_data_for_relative_learning(
            raw_filename, **data_prep_params
        )

        # Train and evaluate the model
        _, _, _, _, mse_list, _ = train_and_evaluate(
            model, df, feature_dicts, optimizer, loss_function, 
            num_epochs=current_params['epochs'], 
            batch_size=current_params['batch_size'], 
            patience=current_params['patience']
        )

        # Calculate average MSE
        avg_mse = sum(mse_list) / len(mse_list)
        print(f"Average MSE: {avg_mse}")

        if avg_mse < best_mse:
            best_mse = avg_mse
            best_params = params

    print(f"Best parameters: {best_params}")
    print(f"Best MSE: {best_mse}")

    return best_params, best_mse

if __name__ == "__main__":
    raw_filename = "/notebooks/data/cal_abs.csv"
    best_params, best_mse = run_random_search(raw_filename, n_iter=10)


Testing parameters: {'weight_decay': 2, 'radius': 4, 'p_dropout': 0.5, 'learning_rate': 4, 'fingerprint_dim': 70}
Epoch 1/2 done. Average Loss: 6.928930646494815
Validation Loss: 5.687583470344544, Validation R2: -0.000903499488323134
Epoch 2/2 done. Average Loss: 6.483275673891368
Validation Loss: 5.685483813285828, Validation R2: -0.0004999166091472218
Epoch 1/2 done. Average Loss: 7.015893459320068
Validation Loss: 5.711077094078064, Validation R2: -0.000480955704226918
Epoch 2/2 done. Average Loss: 6.728467746784813
Validation Loss: 5.703928256034851, Validation R2: 6.046596918909963e-05
Epoch 1/2 done. Average Loss: 6.937916441967613
Validation Loss: 5.761222624778748, Validation R2: -0.0006772444678580097
Epoch 2/2 done. Average Loss: 6.659564865262885
Validation Loss: 5.758454728126526, Validation R2: -0.00032451593068737183
Epoch 1/2 done. Average Loss: 5.5896769934578945
Validation Loss: 4.574968099594116, Validation R2: -0.0013771275196823723
Epoch 2/2 done. Average Loss: 5.3

KeyboardInterrupt: 

In [ ]:
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_squared_error
from concurrent.futures import ThreadPoolExecutor, as_completed

def random_search(param_distributions, n_iter):
    param_list = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=42))
    return param_list

def evaluate_params(raw_filename, fixed_params, params):
    # Combine fixed and variable parameters
    current_params = {**fixed_params, **params}

    # Separate parameters for data preparation and training
    data_prep_params = {k: v for k, v in current_params.items() if k not in ['patience', 'epochs', 'batch_size']}
    
    # Prepare model and data with current hyperparameters
    model, optimizer, loss_function, df, feature_data, feature_filename ,remained_df = prepare_model_and_data_for_relative_learning(
        raw_filename, **data_prep_params
    )

    # Train and evaluate the model
    _, _, _, _, mse, _ = train_and_evaluate(
        model, df, feature_dicts, optimizer, loss_function,
        num_epochs=current_params['epochs'],
        batch_size=current_params['batch_size'],
        patience=current_params['patience']
    )

    return mse, params

def run_random_search(raw_filename, n_iter=10):
    # Define the hyperparameter space
    param_distributions = {
        'radius': [2, 3, 4, 5, 6],
        'fingerprint_dim': [30, 70, 150, 210, 300],
        'p_dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
        'weight_decay': [2, 3, 4, 5, 6],
        'learning_rate': [2, 3, 4, 5]
    }

    # Fixed parameters
    fixed_params = {
        'T': 2,
        'batch_size': 50,
        'epochs': 2,
        'patience': 30
    }

    # Generate random hyperparameter combinations
    param_list = random_search(param_distributions, n_iter)

    best_mse = float('inf')
    best_params = None

    # Use ThreadPoolExecutor for parallel execution
    with ThreadPoolExecutor() as executor:
        futures = {executor.submit(evaluate_params, raw_filename, fixed_params, params): params for params in param_list}

        for future in as_completed(futures):
            mse, params = future.result()
            print(f"Tested parameters: {params} with MSE: {mse}")

            if mse < best_mse:
                best_mse = mse
                best_params = params

    print(f"Best parameters: {best_params}")
    print(f"Best MSE: {best_mse}")

    return best_params, best_mse

if __name__ == "__main__":
    raw_filename = "/notebooks/data/cal_abs.csv"
    best_params, best_mse = run_random_search(raw_filename, n_iter=10)



Epoch 1/2 done. Average Loss: 3843275.279949387
Validation Loss: 5.833428541819255, Validation R2: -0.0005009181857305589
Epoch 1/2 done. Average Loss: 6.699800431728363
Epoch 1/2 done. Average Loss: 10.194848457972208
Epoch 1/2 done. Average Loss: 10.701582491397858
Validation Loss: 6.301166375478108, Validation R2: -0.08060439781777679
Validation Loss: 5.7690502007802325, Validation R2: 0.01274767489422704
Validation Loss: 5.784976005554199, Validation R2: 0.010082194152565976
Epoch 2/2 done. Average Loss: inf
Validation Loss: 6.004336039225261, Validation R2: -0.030921392801456393
Epoch 1/2 done. Average Loss: 7.2936053077379865
Epoch 1/2 done. Average Loss: 7.150484720865886
Epoch 1/2 done. Average Loss: 7.376455545425415
Validation Loss: 5.816035032272339, Validation R2: 0.003083776178861658
Validation Loss: 5.8460477987925215, Validation R2: -0.002884994812739003
Validation Loss: 5.834096908569336, Validation R2: -0.0006585954257555537
Epoch 2/2 done. Average Loss: 6.470349033673

In [16]:
unique_id = str(uuid.uuid4())[:8]
unique_id

'716bf681'